In [ ]:
# Notebook inference entrypoint. Training stays in train_predictor.py.
import os
import sys

cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.append(cwd)

kronos_dir = os.path.join(cwd, 'Kronos')
if os.path.isdir(kronos_dir) and kronos_dir not in sys.path:
    sys.path.append(kronos_dir)

import train_predictor as tp

def _resolve_existing_path(filename):
    candidates = [
        os.path.join(kronos_dir, filename),
        os.path.join(cwd, filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return candidates[0]

def _normalize_date_arg(value):
    if value is None:
        return None
    return str(value)

tokenizer_json_path = _resolve_existing_path('kronos_tokenizer.json')
predictor_json_path = _resolve_existing_path('kronos_predictor.json')

def train_and_save(datasources=None, start_date=None, end_date=None):
    if datasources is None:
        datasources = tp.DEFAULT_DATASOURCES
    else:
        datasources = dict(datasources)

    infer_start = _normalize_date_arg(start_date) or tp.VAL_START
    infer_end = _normalize_date_arg(end_date) or tp.VAL_END

    tp.set_seed(tp.SEED)
    device = tp.get_device()

    tokenizer, tokenizer_stats, _ = tp.load_tokenizer_json(
        tokenizer_json_path,
        map_location=device,
    )
    tokenizer.eval()
    for p in tokenizer.parameters():
        p.requires_grad = False

    predictor, predictor_payload = tp.load_predictor_json(
        predictor_json_path,
        map_location=device,
    )
    predictor.eval()

    feature_cols = predictor_payload['feature_cols']
    seq_len = int(predictor_payload['seq_len'])
    table = datasources['bar1m']

    infer_instruments = tp.pool(infer_start, infer_end)
    if tp.MAX_INFER_INSTRUMENTS is not None:
        infer_instruments = infer_instruments[:tp.MAX_INFER_INSTRUMENTS]

    Xva, Sva, keys_df, last_closes = tp.build_predictor_dataset(
        table=table,
        sd=infer_start,
        ed=infer_end,
        instruments=infer_instruments,
        feature_cols=feature_cols,
        seq_len=seq_len,
        stats=tokenizer_stats,
        max_windows=tp.MAX_INFER_WINDOWS,
        instrument_chunk_size=tp.INSTRUMENT_CHUNK_SIZE,
        sample_stride=tp.SAMPLE_STRIDE,
        return_last_close=True,
    )
    infer_loader = tp.make_inference_dataloader(Xva, Sva, last_closes, tp.BATCH)

    score_df = tp.predict_score_dataframe(
        model=predictor,
        tokenizer=tokenizer,
        loader=infer_loader,
        keys_df=keys_df,
        stats=tokenizer_stats,
        feature_cols=feature_cols,
        device=device,
    )
    return score_df

datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
start_date = '2024-01-01'
end_date = '2024-06-30 23:59:59'
score_df = train_and_save(datasources, start_date, end_date)
score_df
